In [1]:
example = {"instruction": "Given a cue word, your task is to generate a comprehensive list of words associated with the cue word. Aim to cover as many relevant contexts, uses, and meanings as possible without repeating similar concepts. List a target of 70 to 84 words that together provide a broad and insightful representation of all significant associations. Focus on revealing both common and unique aspects related to the cue word to ensure a balanced and thorough exploration of potential associations. Words should be distinct from each other. Your response shall only be the list of associated words. Do not generate words conditioned on the presence of other words but rather focus on the cue word itself.", "input": "mosquito", "output": "bite, bug, itch, buzz, malaria, insect, blood, net, fly, annoying, pest, summer, ouch, itchy, buzzing, repellent, small, swat, irritating, gnat, netting, camping, midge, proboscis, river, pain, lump, sting, flight, disease, spray, slap, swamp, fever, allergy, annoyance, worthless, nest, crunchy, smack, huge  in  canada, dead, amazonian, insect  bite, awake, tropical, water, female, anopheles, coast, valentine, doug, tent, jungle, whine, bumblebee, bored, nozzle, blood  sucker, noisy, nasty, skin, vampire, torment, hawk, ear, itchy  welt, pinch, needle, dengue, africa, bloodsucker, annoying  bug, mosquito  net, australia, horrible, kill, ugly, genetics", "system": "You are a sophisticated language model designed to explore word associations comprehensively."}

In [2]:
example.keys()

dict_keys(['instruction', 'input', 'output', 'system'])

In [3]:
import jsonlines
import os 
from pathlib import Path

In [4]:
with jsonlines.open('alpaca_data_swow_us.jsonl', mode='r') as reader:
    alpaca_data = [obj for obj in reader]


In [5]:
# remove elements if output value contain nan
alpaca_data_new = []
for data in alpaca_data:
    output_content = data['output']
    if not output_content.startswith('nan,'):
        alpaca_data_new.append(data)

print(len(alpaca_data_new) - len(alpaca_data))
alpaca_data = alpaca_data_new


-31092


In [6]:
len(alpaca_data)

1197008

In [7]:
# cluster by input key
def cluster_by_input_key(data, input_key):
    clustered = {}
    for item in data:
        key = item.get(input_key)
        if key not in clustered:
            clustered[key] = []
        clustered[key].append(item)
    return clustered

clustered_data = cluster_by_input_key(alpaca_data, 'input')

In [8]:
# check how many keys are in the clustered data
num_keys = len(clustered_data)
print(f"Number of unique keys in clustered data: {num_keys}")

Number of unique keys in clustered data: 12281


In [9]:
# randomly shuffle and cut into 20% chunks, get 5 rounds
from copy import deepcopy
import random
from tqdm.auto import tqdm

shuffled_keys = list(clustered_data.keys())
# set random seed for splitting train and test 
TRAIN_TEST_SEED = 1234
random.seed(TRAIN_TEST_SEED)
random.shuffle(shuffled_keys)
# split into 80% train and 20% test
split_idx = int(0.8 * len(shuffled_keys))
train_keys = shuffled_keys[:split_idx]
test_keys = shuffled_keys[split_idx:]

# only use train keys for chunking
shuffled_keys = deepcopy(train_keys)
idx_dict = {
    0: '25_percent_train',
    1: '50_percent_train',
    2: '75_percent_train',
    3: '100_percent_train',
}

seed_lst = [42, 43, 44, 45, 46]
for i in tqdm(seed_lst):
    # shuffle the key order
    # set seed
    random.seed(i)
    random.shuffle(shuffled_keys)
    # cut into 25% chunks
    chunk_size = len(shuffled_keys) // 4
    chunks = [shuffled_keys[i * chunk_size:(i + 1) * chunk_size] for i in range(4)]

    folder_name = f"participant_swow_collection_us_seed_{i + 1}"
    # create folder if not exists
    os.makedirs(folder_name, exist_ok=True)
    # create cue_meta_info.json
    cue_meta_info = {
        "seed": i,
        "cue_info": {
            "25%": [],
            "50%": [],
            "75%": [],
            "100%": [],
            'test': test_keys
        }
    }
  

    for chunk_idx, chunk in enumerate(tqdm(chunks, leave=False)):
        # construct back the chunked data
        chunked_data = []
        for key in chunk:
            chunked_data.extend(clustered_data[key])
            cue_meta_info["cue_info"][f"{(chunk_idx + 1) * 25}%"].append(key)
        # shuffle 
        random.shuffle(chunked_data)
        # chunk_data_name
        chunk_data_name = f"{folder_name}/train/chunk_{idx_dict[chunk_idx]}.jsonl"
        # create folder if not exists
        Path(f"{folder_name}/train").mkdir(parents=True, exist_ok=True)
        with jsonlines.open(chunk_data_name, mode='w') as writer:
            writer.write_all(chunked_data)
    # save test set
    test_data = []
    for key in test_keys:
        test_data.extend(clustered_data[key])
    # no shuffle for test set
    test_data_name = f"{folder_name}/test/test.jsonl"
    # create folder if not exists
    Path(f"{folder_name}/test").mkdir(parents=True, exist_ok=True)
    with jsonlines.open(test_data_name, mode='w') as writer:
        writer.write_all(test_data)    
    
    # create cue_meta_info.json
    with jsonlines.open(f"{folder_name}/cue_meta_info.jsonl", mode='w') as writer:
        writer.write(cue_meta_info)


  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

In [14]:
# split into 4 chunks
chunk_size = len(alpaca_data) // 4
chunks = [alpaca_data[i:i + chunk_size] for i in range(0, len(alpaca_data), chunk_size)]


In [15]:
# mkdir swow_us
os.makedirs('swow_us', exist_ok=True)

# save each chunk as jsonl
for i, chunk in enumerate(chunks):
    with jsonlines.open(f'swow_us/chunk_{i}.jsonl', mode='w') as writer:
        writer.write_all(chunk)


In [ ]:
# rsync -chavP /home/sukaih/Documents/Huahua_project/SWOW_US_gen/participant_swow_collection_us/* sukaih@unimeb:/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/data/03_primary/participant_swow_collection_us/ 